# Module F — Genetics

## How much of Alzheimer's risk can we read off the genome?

> ⚠️ **Real published variant effects; simulated genotypes. Section 1 explains which is which.**

### What you will be able to do by the end

1. read a GWAS Catalog table: risk allele, odds ratio, p-value, mapped gene
2. explain why genetics uses p < 5×10⁻⁸ instead of p < 0.05
3. build a **polygenic risk score** by hand, from published effect sizes
4. explain why an AUROC of 0.65 is a *good* result in this field, not a failure
5. state clearly why a polygenic score built in one population transfers badly to another

### The data

**Real effects, simulated people.** The variant table is the genuine **GWAS Catalog** record for Alzheimer's disease: 24 real risk variants with their real risk alleles, real allele frequencies, real odds ratios and real p-values, downloaded from EMBL-EBI. *APOE* is in there, with the largest effect, exactly as it should be.

The **genotypes** are simulated. Individual-level AD genotype data is access-controlled everywhere, so each of our 1200 participants had their alleles drawn from the real published frequencies and their disease status drawn from the real published odds ratios. If the model recovers the literature, that is because the literature is what generated it — the point is the method.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Skim section 2, then do 3.2 (build the score) and section 4.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('F'))


---
# 1 · Understand the data

Two tables. One is a summary of the published literature; the other is our simulated cohort.


### 1.1 The real variant table

This is what a **genome-wide association study (GWAS)** produces. Researchers genotype hundreds of thousands of people, test every common variant in the genome for association with the disease, and publish the ones that survive.

| Column | Meaning |
|---|---|
| `rsid` | The variant's catalogue number, e.g. `rs429358`. |
| `gene` | The nearest or most plausible gene. **Nearest ≠ causal.** |
| `risk_allele` | Which of the two DNA letters at that position carries the risk. |
| `risk_allele_freq` | How common that letter is in the studied population. |
| `odds_ratio` | How much one copy multiplies the odds of disease. |
| `p_value` | Evidence against 'this variant has no effect'. |
| `log_odds` | log(odds ratio) — the weight we use to build the score in section 3. |


In [ ]:
df = load_data('F')
variants = load_extra('F')['variants']

print(f'{len(variants)} real risk variants from the GWAS Catalog.')
print(f'{len(df)} simulated participants, {(df.diagnosis == "AD").mean():.0%} of them cases.\n')
display(variants[['rsid', 'gene', 'risk_allele', 'risk_allele_freq', 'odds_ratio', 'p_value']].head(12))


### 1.2 The shape of genetic risk

**Predict before you run:** *APOE* is famous. Will the other 23 variants have effects roughly as big, or much smaller?


In [ ]:
ordered = variants.sort_values('odds_ratio', ascending=False)
plots.plot_importance(ordered['gene'] + ' (' + ordered['rsid'] + ')', ordered['odds_ratio'] - 1.0,
                      title='Effect size of each real AD risk variant',
                      xlabel='odds ratio minus 1 (0 = no effect)')
plt.show()

print('Note the scale. APOE roughly triples the odds. Most of the rest change them by 10-20%.')
print('That is what "polygenic" means: hundreds of tiny effects, not a handful of big ones.')


### 1.3 Why p < 5×10⁻⁸?

A genome-wide scan tests roughly a million independent positions. At the usual p < 0.05, **50,000 variants would pass by pure chance** — every single one a false discovery.

The genetics field's fix is a Bonferroni correction for a million tests: 0.05 / 10⁶ = 5×10⁻⁸. Every variant in our table clears it. This is not statistical pedantry: it is the reason genetics stopped producing irreproducible 'candidate gene' findings in the 2000s and started producing results that replicate.


In [ ]:
rng = np.random.default_rng(0)
n_tests = 1_000_000
null_p = rng.uniform(size=n_tests)   # a million variants with NO real effect at all

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(['p < 0.05', 'p < 5e-8'], [(null_p < 0.05).sum(), max((null_p < 5e-8).sum(), 0)],
       color=['#c0392b', '#2c6fbb'])
ax.set_yscale('symlog')
ax.set_ylabel('false discoveries (log scale)')
ax.set_title('One million variants, none of them real. How many would you "find"?')
for index, count in enumerate([(null_p < 0.05).sum(), (null_p < 5e-8).sum()]):
    ax.text(index, count, f'  {count:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()


### 1.4 ✏️ Your turn — one variant at a time

`0`, `1` or `2` in a genotype column means how many copies of the risk allele that person carries.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change VARIANT to any rsid from the table above.
#   Try rs429358 (APOE) first, then one of the weaker ones.
#   Ask: could you diagnose anybody from this one variant?
# ==========================================================================
VARIANT = 'rs429358'

info = variants[variants.rsid == VARIANT].iloc[0]
print(f"{VARIANT} — near {info['gene']}, risk allele {info['risk_allele']}, "
      f"published odds ratio {info['odds_ratio']:.2f}\n")

rates = 100 * df.groupby(VARIANT)['diagnosis'].apply(lambda values: (values == 'AD').mean())
counts = df.groupby(VARIANT).size()
plots.plot_score_comparison([f'{dose} copies\n(n={counts[dose]})' for dose in rates.index],
                            rates.tolist(), colours=['#2c6fbb'] * len(rates),
                            reference=100 * (df.diagnosis == 'AD').mean(),
                            title=f'Percentage with AD, by number of {VARIANT} risk alleles',
                            ylabel='percent with AD')
plt.show()
print('Dashed line = the cohort average. Even for APOE, plenty of carriers are unaffected')
print('and plenty of non-carriers are affected. Genetic risk is a shift, not a verdict.')


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** compare rs429358 (APOE) with any variant near the bottom of the effect-size plot.
- 🔵 **If you want to write code:** plot the number of risk alleles a person carries in total against their diagnosis rate. That plot is the whole idea of a polygenic score, before any weighting.
- ⚫ **Take home:** look up *APOE* ε2, ε3 and ε4 — the gene has three common forms, and ε2 is *protective*. Our binary coding hides that. How would you encode it properly?


---
# 2 · Quality control

Genetics has its own characteristic failure modes.


### 2.1 Population stratification

Allele frequencies differ between ancestry groups for reasons that have nothing to do with disease. If your cases and controls are drawn from populations in different proportions, **every variant that differs in frequency between those populations looks associated with the disease**. This is the oldest and most dangerous artefact in genetic epidemiology.

Our cohort deliberately contains two groups with slightly shifted frequencies, so you can see the effect.


In [ ]:
print(df.groupby('ancestry')['diagnosis'].value_counts(normalize=True).round(3))
print()

genotype_columns = [column for column in df.columns if column.startswith('rs')]
difference = (df[df.ancestry == 'reference'][genotype_columns].mean()
              - df[df.ancestry == 'underrepresented'][genotype_columns].mean())

plots.plot_importance(difference.index, difference.values,
                      title='Difference in average risk-allele count between the two ancestry groups',
                      xlabel='mean dose difference (reference minus underrepresented)')
plt.show()
print('These differences are ancestry, not disease. A model cannot tell them apart on its own.')


🧠 **Think first:** Could you fix population stratification by simply adding `ancestry` as a feature to the model?

<details>
<summary>Click for one good answer</summary>

It helps the *prediction*, and it does not fix the *science*. Adding ancestry lets the model stop confusing group membership with disease, so the score improves. But the published effect sizes we used as weights were themselves estimated in a mostly European cohort, so they may be wrong elsewhere — and no amount of adjustment inside our cohort can repair a weight that was measured in a different population.

Real GWAS handle this with **principal components of genome-wide genotype**, included as covariates: a continuous, data-driven summary of ancestry rather than a self-reported label. That controls the artefact well. It still does not make a European-derived score transferable, which is the point of section 3.4.

</details>


### 2.2 ✏️ Your turn — what "nearest gene" hides

GWAS finds *positions*, not genes. The `gene` column is usually just whatever gene is closest, and the causal variant may act on something else entirely — sometimes hundreds of kilobases away. The textbook example is *FTO* and obesity, where the signal turned out to regulate *IRX3*, a different gene.

This cell just prints the table sorted however you like — a reminder to read the columns critically.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Sort by different columns and look at what changes:
#     'p_value'          -> strongest statistical evidence
#     'odds_ratio'       -> biggest biological effect
#     'risk_allele_freq' -> how many people carry it
#   Do the same variants come top under all three? Which ranking
#   would a drug company use, and which would a clinician?
# ==========================================================================
SORT_BY = 'p_value'
ASCENDING = True

display(variants.sort_values(SORT_BY, ascending=ASCENDING)[
    ['rsid', 'gene', 'risk_allele', 'risk_allele_freq', 'odds_ratio', 'p_value', 'reported_trait']].head(12))

plots.plot_scatter(variants['risk_allele_freq'], variants['odds_ratio'],
                   xlabel='how common the risk allele is', ylabel='odds ratio',
                   title='Common variants have small effects; that is not a coincidence')
plt.show()
print('Strong-effect variants get selected against over evolutionary time, so they stay rare.')
print('Common variants survive precisely because their effects are small. Hence: polygenic.')


### 2.3 QC verdict

**Usable, with the field's standing caveats.** Ancestry is in the table and must be either adjusted for or reported. The gene labels are approximate. And the whole enterprise is calibrated on people of European ancestry, which section 4 makes concrete.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


---
# 3 · Build a polygenic risk score

A **polygenic risk score (PRS)** is one of the simplest useful models in all of biomedicine: for each variant, multiply how many risk alleles you carry by the published log-odds, and add it all up.

    PRS = Σ  (number of risk alleles)  ×  log(odds ratio)

That is it. No training, no fitting — the weights come from the published literature. We build it by hand so you can see there is no magic in it.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('F')
variants = load_extra('F')['variants']
print(f'{len(variants)} real GWAS variants, {len(df)} simulated participants. Ready for section 3.')


### 3.1 The simplest model — count APOE alleles


In [ ]:
y = (df['diagnosis'] == 'AD').astype(int)

X_tr, X_te, y_tr, y_te = split_data(df[['rs429358']], y)
apoe_only = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)
plots.plot_roc_pr(y_te, train_model('logistic', X_tr, y_tr).predict_proba(X_te)[:, 1],
                  title='APOE alone')
plt.show()
print(f"APOE alone: AUROC {apoe_only['auroc']:.3f}")


### 3.2 ✏️ Your turn — build the score yourself

Change how many variants go into the score and watch what it buys you. Adding weak variants helps a little and then stops helping — which is why real PRS use hundreds of thousands of variants and *still* only reach modest accuracy.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change N_VARIANTS and re-run. Try 1, 3, 10, 24.
#   Also try INCLUDE_APOE = False to see what the *rest* of the
#   genome contributes once the famous gene is removed.
# ==========================================================================
N_VARIANTS = 24
INCLUDE_APOE = True

chosen = variants.sort_values('p_value')
if not INCLUDE_APOE:
    chosen = chosen[~chosen.gene.isin(['APOE', 'TOMM40', 'NECTIN2', 'APOC1'])]
chosen = chosen.head(N_VARIANTS)

# The polygenic score, written out in full. This is the entire method.
score = np.zeros(len(df))
for _, variant in chosen.iterrows():
    score += df[variant['rsid']].to_numpy() * variant['log_odds']
df['prs'] = score

print(f'Score built from {len(chosen)} variants: {", ".join(chosen.gene.head(6))}...\n')

plots.plot_by_group(df, 'prs', 'diagnosis',
                    title=f'Polygenic risk score from {len(chosen)} variants — note the overlap')
plt.show()

# Risk by decile of the score: the plot that gets shown to patients.
decile = pd.qcut(df['prs'], 10, labels=False, duplicates='drop')
risk_by_decile = 100 * df.assign(decile=decile).groupby('decile')['diagnosis'].apply(
    lambda values: (values == 'AD').mean())
plots.plot_score_comparison([f'{int(d) + 1}' for d in risk_by_decile.index], risk_by_decile.tolist(),
                            colours=['#2c6fbb'] * len(risk_by_decile),
                            reference=100 * y.mean(),
                            title='Percentage with AD, by tenth of the polygenic score',
                            ylabel='percent with AD')
plt.show()

X_tr, X_te, y_tr, y_te = split_data(df[['prs']], y)
prs_metrics = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)
print(f"PRS from {len(chosen)} variants: AUROC {prs_metrics['auroc']:.3f}")


### 3.3 The ladder

Four models, from a single gene to everything we have.


In [ ]:
genotype_columns = [column for column in df.columns if column.startswith('rs')]
ladder = {
    'APOE only': df[['rs429358']],
    'PRS only': df[['prs']],
    'PRS + age + sex': df[['prs', 'age', 'sex']],
    'all 24 variants + age + sex': df[genotype_columns + ['age', 'sex']],
    'age + sex only': df[['age', 'sex']],
}
scores = {}
for name, table in ladder.items():
    X_tr, X_te, y_tr, y_te = split_data(table, y)
    scores[name] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

plots.plot_score_comparison(list(scores), list(scores.values()), reference=0.5,
                            colours=['#8a8a8a', '#2c6fbb', '#2c6fbb', '#2c6fbb', '#e08214'],
                            title='Genetic risk models — read the y-axis carefully',
                            ylabel='AUROC')
plt.show()
for name, value in scores.items():
    print(f'  {name:<30s} {value:.3f}')


🧠 **Think first:** The best model here reaches an AUROC in the low 0.7s. In module C the blood test reached 0.9. Is genetics just worse?

<details>
<summary>Click for one good answer</summary>

It is answering a **completely different question**, and 0.7 is a strong result for it. A blood biomarker measures pathology *that is already happening in the brain of a person who came to a memory clinic*. A polygenic score is measured at birth, decades before anything happens, in someone with no symptoms at all. Predicting a lifetime outcome from a genome is a far harder problem — real published AD polygenic scores reach AUROC around 0.65–0.75 including *APOE*, and the field considers that genuinely useful for stratifying trial recruitment. **Judge a score against what the alternative was, not against another module.**

</details>


### 3.4 🔵 Your turn to write code — does the score transfer?

The most important equity question in modern genetics. Our PRS weights come from studies conducted overwhelmingly in people of European ancestry. Allele frequencies and linkage patterns differ between populations, so the same weights do not carry the same meaning elsewhere.

Fill in the `# TODO`: score the two ancestry groups separately.


In [ ]:
# TODO (🔵): compute the PRS-only AUROC separately within each ancestry group.
#   1. loop over df.groupby('ancestry')
#   2. inside each group, split_data on group[['prs']] and the group's AD indicator
#   3. store the AUROC in transfer[name]
transfer = {}

if transfer:
    plots.plot_score_comparison(list(transfer), list(transfer.values()), reference=0.5,
                                colours=['#2c6fbb', '#c0392b'],
                                title='The same polygenic score, evaluated in two populations',
                                ylabel='AUROC')
    plt.show()
else:
    print('Fill in the TODO above. The solutions notebook has a worked version.')


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 3.2 set `INCLUDE_APOE = False` and see how much of the genetic signal was one gene.
- 🔵 **If you want to write code:** complete 3.4, then repeat it for the full 24-variant model instead of the PRS.
- ⚫ **Take home:** real PRS use hundreds of thousands of variants with shrinkage methods (LDpred, PRS-CS) that account for correlation between nearby variants. Read how one of them works.


---
# 4 · Read the results

Genetics results demand an unusually careful reading, because the temptation to over-interpret them is unusually strong.


### 4.1 The standard views


In [ ]:
X = df[['prs', 'age', 'sex']]
X_train, X_test, y_train, y_test = split_data(X, y)
final_model = train_model('logistic', X_train, y_train)
probability = final_model.predict_proba(X_test)[:, 1]
predicted = (probability >= 0.5).astype(int)
final_metrics = evaluate(final_model, X_test, y_test)

plots.plot_confusion(y_test, predicted, labels=('no AD', 'AD'), title='PRS + age + sex')
plt.show()
plots.plot_roc_pr(y_test, probability, title='Polygenic risk model, held-out participants')
plt.show()
plots.plot_calibration(y_test, probability)
plt.show()
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


### 4.2 Shapley values — what is the score made of?

Exact Shapley values on a small feature set, computed by enumerating all 2⁵ = 32 coalitions. Here they separate the genetic contribution from the demographic one.


In [ ]:
from interpret import shapley_values, shapley_importance, baseline_prediction

top_variants = variants.sort_values('p_value').head(4)['rsid'].tolist()
explain_columns = ['age', 'prs'] + top_variants
explain_frame = df[explain_columns]
X_tr2, X_te2, y_tr2, y_te2 = split_data(explain_frame, y)
explain_model = train_model('random_forest', X_tr2, y_tr2)

shap_frame = shapley_values(explain_model, X_te2.head(60), X_tr2, features=explain_columns)
importance = shapley_importance(shap_frame)
plots.plot_importance(importance.index, importance.values,
                      title='What drives the predicted risk (exact Shapley values)',
                      xlabel='mean |contribution| to predicted probability')
plt.show()

PERSON = 0
one = shap_frame.iloc[PERSON].sort_values()
plots.plot_importance(one.index, one.values,
                      title=f'Participant {X_te2.index[PERSON]}: what raised and lowered their risk',
                      xlabel='contribution to predicted probability')
plt.show()
print(f'Cohort average risk {baseline_prediction(explain_model, X_tr2):.3f} '
      f'{one.sum():+.3f} = {baseline_prediction(explain_model, X_tr2) + one.sum():.3f}')


### 4.3 ✏️ Your turn — where would you set the threshold?

PRS are not used to diagnose. They are used to *stratify*: to decide who to invite into a prevention trial, or who to monitor more closely. That means picking a top slice of the distribution.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   TOP_PERCENT is how much of the population you would flag as high risk.
#   Try 1, 5, 10, 25.
#   For each: how many real future cases are inside your flagged group,
#   and how many did you miss? This is exactly the trade-off a trial
#   recruiter faces with a fixed budget.
# ==========================================================================
TOP_PERCENT = 10

cutoff = np.percentile(df['prs'], 100 - TOP_PERCENT)
flagged = df['prs'] >= cutoff
cases = df['diagnosis'] == 'AD'

captured = (flagged & cases).sum() / cases.sum()
enrichment = cases[flagged].mean() / cases.mean()

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist([df.loc[~cases, 'prs'], df.loc[cases, 'prs']], bins=35, stacked=True,
        color=['#cccccc', '#e08214'], label=['no AD', 'AD'])
ax.axvline(cutoff, color='#c0392b', linewidth=2)
ax.annotate(f'top {TOP_PERCENT}%', (cutoff, ax.get_ylim()[1] * 0.85),
            xytext=(6, 0), textcoords='offset points', color='#c0392b', fontsize=9)
ax.set_xlabel('polygenic risk score'); ax.set_ylabel('number of people')
ax.set_title(f'Flagging the top {TOP_PERCENT}% captures {captured:.0%} of all cases')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f'You would invite {flagged.sum()} of {len(df)} people.')
print(f'Among them, {cases[flagged].mean():.1%} develop AD, versus {cases.mean():.1%} in the whole cohort.')
print(f'That is a {enrichment:.2f}x enrichment — the number a trial designer actually cares about.')
print(f'But you also missed {(cases & ~flagged).sum()} future cases who scored below the line.')


### 4.4 Does it work equally well for everybody?


In [ ]:
check = df.loc[X_test.index].copy()
check['correct'] = (predicted == y_test.to_numpy()).astype(int)
check['age_band'] = pd.cut(check['age'], [54, 68, 76, 84, 96], labels=['<68', '68-76', '76-84', '84+'])

for subgroup in ['ancestry', 'sex', 'age_band']:
    plots.plot_subgroup_errors(check.dropna(subset=[subgroup]), subgroup, 'correct',
                               title=f'Proportion correct by {subgroup}')
    plt.show()


### 4.5 Your headline result


In [ ]:
plots.plot_score_comparison(list(final_metrics), list(final_metrics.values()), reference=0.5,
                            colours=['#2c6fbb'] * 5,
                            title='Module F — polygenic score + age + sex, held-out participants',
                            ylabel='score')
plt.show()
print(f'{len(variants)} real GWAS Catalog variants; {len(df)} simulated participants.')
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')
print('\nRemember: 0.70 here is a good result. Compare it to what you could have said')
print('about the same person with no information at all, not to another module.')


### 4.6 What would have to be true before this touched a patient?

1. **The genotypes are simulated.** The variant effects are real and published; the people are not.
2. **A PRS is not a diagnosis and never becomes one.** Someone in the top 1% may well never develop dementia. Someone in the bottom 1% may. It shifts a probability.
3. **Ancestry bias is not a technical footnote.** Around 80% of GWAS participants to date are of European ancestry. A score built from those studies works less well elsewhere — and, as 3.4 shows, the groups it works worst for are also the groups where we can least reliably measure how badly it works. Deploying such a score at scale would widen an existing health gap while looking objective.
4. **There is no treatment to offer.** Risk information without an intervention is a burden, not a benefit. This is why *APOE* genotyping is not routinely offered outside research.
5. **Genetic information is not only about the person tested.** It is about their siblings and children, who did not consent.

---

### 🧠 Final question for the group discussion

You can be genotyped for about €50, once, forever. **Should everyone get an AD polygenic score at birth?** If not now, what would have to change first?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 4.3, pick the `TOP_PERCENT` you would use to recruit a prevention trial and justify it.
- 🔵 **If you want to write code:** combine the polygenic score with a biomarker-like feature and see whether they add to each other. (Genes and biomarkers measure different stages of the same process.)
- ⚫ **Take home:** read about the ethics of returning *APOE* results to research participants — including the REVEAL study, which actually measured what happens to people who are told.
